# 14 - Concern 5: validation on a second dataset (fraudulent job postings, EMSCAD)

**Reviewer concern.** All results come from one corpus. A second dataset is needed to show that the method,
and not a property of one corpus, drives the findings.

**Why this dataset (the thesis's four corpora compared).** EMSCAD (Vidros et al., 2017; 17,880 postings,
866 fraudulent) is the only other thesis corpus on which every element of the paper can be tested:

- it is **text organised in named sections** (title, company profile, description, requirements, benefits),
  so section-tag subtraction, the tag-direction projection and Proposition 1 all apply;
- unlike LinkedIn, the **set of sections varies** between postings (10 presence patterns) and does so with the
  class: a company profile is present in 84% of legitimate but 32% of fraudulent postings. This is exactly the
  situation in which the paper predicts the tag term carries a *composition* signal. Here that prediction is
  testable: STE should beat the plain mean, and adding composition features explicitly should close the gap;
- it is **heavily imbalanced** (4.84% fraudulent), a hard test for the weakest-metric OWA objective;
- it is **large**, so confidence intervals are tight.

InstaFake has eight numeric fields and no text, and BotHawk has short free-text fields without a section
structure; neither can test the embedding contributions.

**Two protocols.** After whitespace normalisation, 2,001 postings (11.2%; 153 of them fraudulent) repeat the five
text sections of an earlier posting. A random split then places copies of test postings in the training set.
- `dedup`: one copy of each duplicate group is kept (15,879 postings, 713 fraudulent) - the leakage-free main analysis;
- `full`: all 17,880 postings, as in most published work - run to measure how much duplicates inflate results.

**Pipeline.** Identical to the paper: frozen RoBERTa section embeddings, logistic regression, HGS with the
fuzzy-OWA objective and gap penalty, matched-budget random search, stratified 70/30 splits, seeds 42, 7, 123.
The 15 LinkedIn counts are replaced by the posting metadata (telecommuting, logo, screening questions,
salary/department/location present, employment type, experience, education, function, industry).

**Runtime.** GPU for about 15-30 minutes once (section embeddings, cached). Then CPU: each HGS run takes about
15-60 minutes because the training split is five times larger than on LinkedIn; folds run in parallel on all
available cores, and every evaluation is saved, so the plan can span several sessions.
**Outputs:** `results_emscad.csv`, `emscad_report.txt`, predictions in `predictions/`.

## Setup

In [ ]:
!pip -q install transformers scikit-learn scipy
import os, re, json, time, math, random, itertools, hashlib, warnings, pickle
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATA_DIR = os.environ.get("EMSCAD_DATA_DIR", "/content/drive/MyDrive/fake_profile_paper" if IN_COLAB else os.path.abspath("../data"))
OUT_DIR = os.environ.get("EMSCAD_OUT_DIR", "/content/drive/MyDrive/fake_profile_paper/emscad_validation" if IN_COLAB else os.path.abspath("../results/emscad"))
os.makedirs(OUT_DIR, exist_ok=True)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
                             average_precision_score, matthews_corrcoef, balanced_accuracy_score)
from joblib import Parallel, delayed
N_JOBS = max(1, min(6, os.cpu_count() or 1))
PRED_DIR = os.path.join(OUT_DIR, "predictions"); os.makedirs(PRED_DIR, exist_ok=True)
CACHE_DIR = os.path.join(OUT_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)
print("data:", DATA_DIR, "| results:", OUT_DIR, "| folds in parallel:", N_JOBS)

## Data, duplicates and section presence

In [ ]:
EMSCAD_FILE = os.path.join(DATA_DIR, "fake_job_postings.csv")
EMSCAD_SHA256 = "cec194ca0185a973d9c339d7d83c43d80fb65b2efdcd5f5a7a12c80c85cae3e2"   # 17,880 rows, 866 fraudulent
if not os.path.exists(EMSCAD_FILE):
    raise FileNotFoundError("Put fake_job_postings.csv (Kaggle: shivamb/real-or-fake-fake-jobposting-prediction) in " + DATA_DIR)
_h = hashlib.sha256(open(EMSCAD_FILE, "rb").read()).hexdigest()
print("checksum:", "matches the release used here" if _h == EMSCAD_SHA256 else "differs (%s) - check the counts below" % _h)
df = pd.read_csv(EMSCAD_FILE)
assert len(df) == 17880 and int(df.fraudulent.sum()) == 866, "unexpected counts: %d rows, %d fraudulent" % (len(df), df.fraudulent.sum())

SECS = ["title", "company_profile", "description", "requirements", "benefits"]
TAGS = ["Title", "Company profile", "Description", "Requirements", "Benefits"]
def clean(t):
    return "" if pd.isna(t) else re.sub(r"\s+", " ", str(t)).strip()      # placeholders such as #URL_..# are kept
TEXT = np.array([[clean(v) for v in df[s]] for s in SECS], dtype=object).T
P_ALL = np.array([[len(t) > 0 for t in row] for row in TEXT])
Y_ALL = df.fraudulent.values.astype(int)
KEY = pd.Series(["||".join(r) for r in TEXT])
ROWSETS = {"full": np.arange(len(df)), "dedup": np.where(~KEY.duplicated().values)[0]}
for k, rows in ROWSETS.items():
    print("%-5s postings %5d | fraudulent %3d (%.2f%%)" % (k, len(rows), Y_ALL[rows].sum(), 100 * Y_ALL[rows].mean()))
print("\nsection present:   legitimate   fraudulent")
for j, s in enumerate(SECS):
    print("  %-16s %6.1f%%     %6.1f%%" % (s, 100 * P_ALL[Y_ALL == 0, j].mean(), 100 * P_ALL[Y_ALL == 1, j].mean()))
PATTERN = np.array(["".join("1" if x else "0" for x in r) for r in P_ALL])
print("distinct presence patterns:", len(set(PATTERN)))

## Section embeddings (GPU once, then cached)

In [ ]:
import torch
from transformers import RobertaTokenizer, RobertaModel
EMB_PATH = os.path.join(OUT_DIR, "emscad_roberta_sections.npz")
if os.path.exists(EMB_PATH):
    z = np.load(EMB_PATH); E_ALL, TAG_E = z["E"], z["T"]
    assert z["sha"].item() == _h, "cached embeddings were computed from a different file"
else:
    if not torch.cuda.is_available():
        raise RuntimeError("No cached embeddings: run this cell once on a GPU runtime (T4 is enough).")
    device = torch.device("cuda")
    tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
    roberta = RobertaModel.from_pretrained("roberta-base").to(device).eval()
    @torch.no_grad()
    def embed_many(texts, batch_size=32):
        """Mean over all tokens (special tokens included, padding excluded), as embed_text() in the LinkedIn notebooks."""
        out = np.zeros((len(texts), 768), np.float32)
        todo = sorted([i for i, t in enumerate(texts) if t], key=lambda i: len(texts[i]))
        for k in range(0, len(todo), batch_size):
            ids = todo[k:k + batch_size]
            enc = tokenizer([texts[i] for i in ids], return_tensors="pt", padding=True, truncation=True, max_length=512).to(device)
            h = roberta(**enc).last_hidden_state.float(); m = enc["attention_mask"].unsqueeze(-1).float()
            out[ids] = ((h * m).sum(1) / m.sum(1)).cpu().numpy()
        return out
    t0 = time.time(); E_ALL = np.zeros((len(df), 5, 768), np.float32)
    for j, s in enumerate(SECS):
        E_ALL[:, j] = embed_many(list(TEXT[:, j])); print("  embedded %-16s %.0fs" % (s, time.time() - t0))
    TAG_E = embed_many(TAGS)
    np.savez_compressed(EMB_PATH, E=E_ALL, T=TAG_E, sha=np.array(_h))
print("section embeddings:", E_ALL.shape, "| tags:", TAG_E.shape)

## HGS, fuzzy OWA and the logistic-regression space (identical to the paper)

In [ ]:
# =============================================================================
#  MODULE : Hunger Games Search (HGS) + Fuzzy OWA Aggregation Anti-Overfitting
#  ---------------------------------------------------------------------------
#  Référence HGS : Yang Y., Chen H., Heidari A.A., Gandomi A.H. (2021)
#                  "Hunger Games Search", ESWA 177, 114864.
#  Référence OWA  : Yager R.R. (1988) "On Ordered Weighted Averaging Aggregation
#                                      Operators in Multi-Criteria Decision Making"
#
#  STRATÉGIE ANTI-OVERFITTING :
#  ---------------------------------------------------------------------------
#  1)  Le test set n'est JAMAIS vu par le fitness (sacré jusqu'à l'évaluation
#      finale). Fitness = K-fold CV stratifiée sur (X_train, y_train).
#  2)  Agrégation FLOUE (OWA pessimiste) des K scores : on donne plus de poids
#      aux PIRES plis → pénalise la variance entre folds → robustesse.
#  3)  Fuzzy multi-metric : fusion F1 ⊕ AUC ⊕ Recall par OWA pondéré (chaque
#      métrique est min-max normalisée).
#  4)  Pénalité de généralisation : fitness final = OWA_cv - λ · (train - μ_cv)
#      → décourage les modèles qui sur-apprennent (gap train-CV élevé).
#  5)  Early-stopping HGS : si pas d'amélioration pendant `patience` itérations,
#      on arrête → moins d'évaluations, moins de risque de surajustement à un
#      pli particulier.
# =============================================================================
import math, random, time
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone
from sklearn.metrics import f1_score, roc_auc_score, recall_score, accuracy_score


# -----------------------------------------------------------------------------
#  1. Fuzzy OWA (Ordered Weighted Average) — Yager 1988
# -----------------------------------------------------------------------------
def owa_weights(n, orness=0.30):
    """
    Génère des poids OWA pour un vecteur de taille n.
        orness ∈ [0,1] :
            • 0.0  → "AND" (min)              très pessimiste
            • 0.5  → moyenne arithmétique
            • 1.0  → "OR"  (max)              très optimiste
        On choisit orness = 0.30 (légèrement pessimiste) : on récompense
        les modèles dont les PIRES folds restent corrects.
    Implémentation : poids exponentiels biaisés vers la fin (queue basse).
    """
    if n <= 1:
        return np.ones(n) / max(n, 1)
    # Distribution exponentielle : alpha contrôle la pente
    # alpha = -ln(orness)  ⇒ pour orness < 0.5, poids concentrés sur les PETITES
    #                       valeurs (premiers éléments du vecteur trié ascending)
    #                       → moyenne pondérée tirée vers le bas = pessimiste.
    alpha = max(-math.log(max(orness, 1e-6)), 1e-6)
    idx = np.arange(n)
    # poids décroissants : w[0] (pire valeur) = max, w[n-1] (meilleure) = min
    w = np.exp(-alpha * idx / max(n - 1, 1))
    return w / w.sum()


def fuzzy_owa(scores, orness=0.30):
    """
    Agrégation OWA d'un vecteur de scores.
    1) Trier les scores en ordre croissant.
    2) Appliquer w_i tel que les premiers (pires) reçoivent plus de poids.
    """
    arr = np.asarray(scores, dtype=float).ravel()
    if arr.size == 0:
        return 0.0
    arr = np.sort(arr)                          # ordre croissant
    w   = owa_weights(arr.size, orness=orness)  # poids biaisés bas
    return float(np.dot(w, arr))


def fuzzy_multi_metric(y_true, y_pred, y_proba=None, orness=0.30):
    """
    Combine F1, Recall et AUC (si y_proba) en un score flou unique via OWA.
    Chaque métrique ∈ [0,1] → comparable.
    """
    binary = len(np.unique(y_true)) == 2
    avg = "binary" if binary else "macro"
    f1  = f1_score(y_true, y_pred, average=avg, zero_division=0)
    rec = recall_score(y_true, y_pred, average=avg, zero_division=0)
    metrics = [f1, rec]
    if y_proba is not None and binary:
        try:
            auc = roc_auc_score(y_true, y_proba)
            metrics.append(auc)
        except Exception:
            pass
    return fuzzy_owa(metrics, orness=orness)


# -----------------------------------------------------------------------------
#  2. Fitness K-fold CV avec fuzzy aggregation + pénalité de gap
# -----------------------------------------------------------------------------
def make_cv_fitness(estimator_factory, X_train, y_train,
                    n_splits=5, seed=42, orness=0.30,
                    gap_penalty=0.50, use_proba=True):
    """
    Renvoie une fonction de fitness `fit(params) -> float` agrégée via OWA flou.

        estimator_factory : callable(params:dict) -> sklearn-compatible model
        n_splits          : nb de folds (5 par défaut)
        orness            : ∈ [0,1], <0.5 = pessimiste, pénalise variance
        gap_penalty       : λ, pénalité du gap (train - μ_cv) ∈ [0,1]
        use_proba         : True si l'estimator expose predict_proba (AUC)

    Le SCORE final renvoyé :
        score = OWA_cv  -  λ · max(0, train_full - mean_cv)
    où :
        OWA_cv     = agrégation floue OWA des K scores fuzzy_multi_metric
        train_full = score fuzzy_multi_metric sur l'ENSEMBLE train re-fit
        gap        = (train_full - mean_cv)  → quantifie l'overfitting
    """
    X_train = np.asarray(X_train) if not hasattr(X_train, "toarray") else X_train
    y_train = np.asarray(y_train)

    def _safe_score(est, X_tr, y_tr, X_va, y_va):
        try:
            est.fit(X_tr, y_tr)
            y_pred = est.predict(X_va)
            y_proba = None
            if use_proba and hasattr(est, "predict_proba"):
                try:
                    y_proba = est.predict_proba(X_va)[:, 1]
                except Exception:
                    y_proba = None
            return fuzzy_multi_metric(y_va, y_pred, y_proba, orness=orness)
        except Exception:
            return 0.0

    def fitness(params):
        # --- K-fold CV stratifiée sur X_train UNIQUEMENT (jamais X_test) ---
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        fold_scores = []
        try:
            est_template = estimator_factory(params)
        except Exception:
            return 0.0

        for tr_idx, va_idx in skf.split(np.zeros(len(y_train)), y_train):
            # sparse-safe indexing
            if hasattr(X_train, "tocsr"):
                Xtr_f, Xva_f = X_train.tocsr()[tr_idx], X_train.tocsr()[va_idx]
            else:
                Xtr_f, Xva_f = X_train[tr_idx], X_train[va_idx]
            ytr_f, yva_f = y_train[tr_idx], y_train[va_idx]
            est = clone(est_template)
            fold_scores.append(_safe_score(est, Xtr_f, ytr_f, Xva_f, yva_f))

        owa_cv  = fuzzy_owa(fold_scores, orness=orness)
        mean_cv = float(np.mean(fold_scores))

        # --- Score train complet pour calcul de gap (overfitting) ---
        est_full = clone(est_template)
        try:
            est_full.fit(X_train, y_train)
            y_pred_tr = est_full.predict(X_train)
            y_proba_tr = None
            if use_proba and hasattr(est_full, "predict_proba"):
                try:
                    y_proba_tr = est_full.predict_proba(X_train)[:, 1]
                except Exception:
                    y_proba_tr = None
            train_full = fuzzy_multi_metric(y_train, y_pred_tr, y_proba_tr, orness=orness)
        except Exception:
            train_full = mean_cv

        gap     = max(0.0, train_full - mean_cv)
        penalty = gap_penalty * gap
        return owa_cv - penalty

    return fitness


# -----------------------------------------------------------------------------
#  3. HGS Optimizer (avec early-stopping pour limiter le risque d'overfit)
# -----------------------------------------------------------------------------
class HGSOptimizer:
    """
    Hunger Games Search — Yang et al. 2021.
    Compatible recherche discrète : search_space = {param: [v1, v2, ...]}.

    Améliorations anti-overfitting :
        • early_stop_patience : arrête HGS si pas d'amélioration sur N itérations
        • Le fitness est typiquement obtenu via make_cv_fitness (fuzzy OWA + gap).
    """
    def __init__(self, search_space, fitness_fn,
                 pop_size=8, max_iter=15,
                 l_h=1e-2, LH=1e4,
                 seed=42, verbose=True,
                 early_stop_patience=4):
        self.space   = search_space
        self.keys    = list(search_space.keys())
        self.dims    = [len(search_space[k]) for k in self.keys]
        self.D       = len(self.keys)
        self.fit     = fitness_fn
        self.N       = pop_size
        self.T       = max_iter
        self.l_h     = l_h
        self.LH      = LH
        self.verbose = verbose
        self.patience = early_stop_patience
        random.seed(seed); np.random.seed(seed)
        self.history = []
        self.best_params_ = None
        self.best_score_  = -np.inf

    def _decode(self, x):
        params = {}
        for i, k in enumerate(self.keys):
            idx = int(min(self.dims[i] - 1, max(0, math.floor(x[i] * self.dims[i]))))
            params[k] = self.space[k][idx]
        return params

    def _evaluate(self, x, cache):
        params = self._decode(x)
        key = tuple(sorted(params.items(), key=lambda kv: kv[0]))
        if key in cache:
            return cache[key], params
        s = self.fit(params)
        cache[key] = s
        return s, params

    def optimize(self):
        X = np.random.rand(self.N, self.D)
        F = np.full(self.N, -np.inf)
        Hunger = np.zeros(self.N)
        cache = {}

        for i in range(self.N):
            F[i], _ = self._evaluate(X[i], cache)

        best_idx = int(np.argmax(F))
        X_best   = X[best_idx].copy()
        F_best   = F[best_idx]
        self.best_params_ = self._decode(X_best)
        self.best_score_  = F_best
        self.history.append(F_best)

        if self.verbose:
            print(f"[HGS] init  best fuzzy-score={F_best:.4f}  params={self.best_params_}")

        no_improve = 0
        for t in range(self.T):
            E_t = 2 * (1 - t / max(1, self.T))
            best_fit  = F.max();  worst_fit = F.min()

            for i in range(self.N):
                if abs(best_fit - worst_fit) < 1e-12:
                    TH = 0
                else:
                    TH = (F[i] - best_fit) / (best_fit - worst_fit + 1e-12) * \
                         random.random() * 2 * (1 - t / max(1, self.T))
                hunger_i = TH if abs(TH) > self.l_h else self.l_h
                Hunger[i] += min(self.LH, abs(hunger_i))

            S_hunger = Hunger.sum() + 1e-12
            W1 = np.zeros(self.N); W2 = np.zeros(self.N)
            for i in range(self.N):
                if random.random() < (Hunger[i] / S_hunger):
                    W1[i] = Hunger[i] * self.N / S_hunger * random.random()
                else:
                    W1[i] = 1.0
                W2[i] = (1 - math.exp(-abs(Hunger[i] - Hunger[best_idx]))) * \
                        random.random() * 2

            improved_iter = False
            for i in range(self.N):
                r1, r2 = random.random(), random.random()
                a = 2 * E_t * r1 - E_t
                R = a
                E = math.copysign(1, random.random() - 0.5) * \
                    (math.exp(-abs(F[i] - F_best) / (abs(F_best) + 1e-12)))

                levy_step = 0.01 * np.random.standard_cauchy(self.D)
                if r1 < self.l_h:
                    X_new = np.random.rand(self.D)
                elif r2 > E:
                    X_new = W1[i] * X_best + R * W2[i] * abs(X_best - X[i])
                else:
                    X_new = W1[i] * X_best - R * W2[i] * abs(X_best - X[i]) + levy_step
                X_new = np.clip(X_new, 0.0, 1.0)

                F_new, params_new = self._evaluate(X_new, cache)
                if F_new > F[i]:
                    X[i] = X_new
                    F[i] = F_new
                    if F_new > F_best:
                        F_best = F_new
                        X_best = X_new.copy()
                        best_idx = i
                        self.best_params_ = params_new
                        self.best_score_  = F_best
                        improved_iter = True
                        if self.verbose:
                            print(f"[HGS] t={t+1:02d}  ↑  fuzzy-score={F_best:.4f}  {params_new}")

            self.history.append(F_best)
            no_improve = 0 if improved_iter else (no_improve + 1)
            if no_improve >= self.patience:
                if self.verbose:
                    print(f"[HGS] early-stop @ t={t+1} (no improvement for {self.patience} iters)")
                break

        if self.verbose:
            print(f"[HGS] DONE   best fuzzy-score={self.best_score_:.4f}  "
                  f"params={self.best_params_}")
        return self.best_params_, self.best_score_, self.history


def hgs_search(search_space, fitness_fn,
               pop_size=8, max_iter=15, seed=42, verbose=True,
               early_stop_patience=4):
    """API simple : best_params, best_score, history = hgs_search(...)"""
    opt = HGSOptimizer(search_space, fitness_fn,
                       pop_size=pop_size, max_iter=max_iter,
                       seed=seed, verbose=verbose,
                       early_stop_patience=early_stop_patience)
    t0 = time.time()
    best_params, best_score, history = opt.optimize()
    print(f"[HGS] elapsed = {time.time()-t0:.1f}s")
    return best_params, best_score, history

In [ ]:
LR_SPACE = {
    "C":            [0.01, 0.05, 0.1, 0.3, 0.5, 1.0, 2.0],   # plafond abaissé
    "penalty":      ["l1", "l2", "elasticnet"],
    "solver":       ["liblinear", "saga", "lbfgs"],
    "class_weight": [None, "balanced"],
    "max_iter":     [500, 1000, 2000],
    "l1_ratio":     [0.1, 0.5, 0.9],
}

VALID_COMBOS = {
    ("l1", "liblinear"), ("l1", "saga"),
    ("l2", "liblinear"), ("l2", "saga"), ("l2", "lbfgs"),
    ("elasticnet", "saga"),
}

def build_estimator(params):
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS:
        # combinaison invalide : on retombe sur (l2, lbfgs)
        p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet":
        p.pop("l1_ratio", None)
    if p["solver"] == "liblinear":
        return LogisticRegression(**p, random_state=42, n_jobs=1)
    return LogisticRegression(**p, random_state=42, n_jobs=-1)


def canonical(params):
    # the model build_estimator() actually fits - used to merge duplicate grid points
    p = dict(params)
    if (p["penalty"], p["solver"]) not in VALID_COMBOS: p["penalty"], p["solver"] = "l2", "lbfgs"
    if p["penalty"] != "elasticnet": p["l1_ratio"] = None
    return tuple(sorted(p.items(), key=lambda kv: kv[0]))

DISTINCT = {}
for combo in itertools.product(*LR_SPACE.values()):
    p = dict(zip(LR_SPACE.keys(), combo)); DISTINCT.setdefault(canonical(p), p)
print("nominal grid:", int(np.prod([len(v) for v in LR_SPACE.values()])),
      "| distinct models:", len(DISTINCT))

In [ ]:
class Counted:
    # wraps a fitness so the number of DISTINCT models evaluated is known exactly
    def __init__(self, fn): self.fn, self.seen = fn, {}
    def __call__(self, params):
        k = canonical(params)
        if k not in self.seen: self.seen[k] = self.fn(params)
        return self.seen[k]
    @property
    def n(self): return len(self.seen)

def random_search(fitness, budget, seed):
    keys = list(DISTINCT.keys()); rng = random.Random(seed); rng.shuffle(keys)
    best_p, best_s = None, -1e9
    for k in keys[:budget]:
        s = fitness(DISTINCT[k])
        if s > best_s: best_p, best_s = DISTINCT[k], s
    return best_p, best_s

def grid_search(fitness):
    best_p, best_s = None, -1e9
    for i, p in enumerate(DISTINCT.values()):
        s = fitness(p)
        if s > best_s: best_p, best_s = p, s
        if (i+1) % 50 == 0: print("    grid", i+1, "/", len(DISTINCT), "best", round(best_s, 4))
    return best_p, best_s

def evaluate(y, proba, thr=0.5):
    yp = (proba >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yp, labels=[0,1]).ravel()
    return dict(
        accuracy=accuracy_score(y, yp),
        precision_pos=precision_score(y, yp, zero_division=0),
        recall_pos=recall_score(y, yp, zero_division=0),
        f1_pos=f1_score(y, yp, zero_division=0),
        f1_macro=f1_score(y, yp, average="macro", zero_division=0),
        precision_w=precision_score(y, yp, average="weighted", zero_division=0),
        recall_w=recall_score(y, yp, average="weighted", zero_division=0),
        f1_w=f1_score(y, yp, average="weighted", zero_division=0),
        roc_auc=roc_auc_score(y, proba), pr_auc=average_precision_score(y, proba),
        mcc=matthews_corrcoef(y, yp), bal_acc=balanced_accuracy_score(y, yp),
        TP=int(tp), FP=int(fp), FN=int(fn), TN=int(tn), n_test=int(len(y)))

In [ ]:
import pickle
CACHE_DIR = os.path.join(OUT_DIR, "cache"); os.makedirs(CACHE_DIR, exist_ok=True)

def _atomic_pickle(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, path)

def _atomic_csv(df, path):
    tmp = path + ".tmp"; df.to_csv(tmp, index=False); os.replace(tmp, path)

class PersistentCounted:
    # fitness wrapper: saves each evaluation to Drive; n = distinct models queried by THIS search
    def __init__(self, fn, cache_path):
        self.fn, self.path = fn, cache_path
        self.cache = pickle.load(open(cache_path, "rb")) if os.path.exists(cache_path) else {}
        self.queried = set()
        if self.cache: print("    resuming from %d cached evaluations" % len(self.cache))
    def __call__(self, params):
        k = canonical(params); self.queried.add(k)
        if k not in self.cache:
            self.cache[k] = self.fn(params); _atomic_pickle(self.path, self.cache)
        return self.cache[k]
    @property
    def n(self): return len(self.queried)

## Statistics

In [ ]:
# ---- statistics shared by notebooks 10-14 ----------------------------------------------------
from scipy import stats as sst

def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j
    out = np.empty(N); out[J] = T
    return out

def delong_test(y, s1, s2):
    """DeLong, DeLong and Clarke-Pearson (1988) test for two correlated ROC areas,
    computed with the fast algorithm of Sun and Xu (2014). Returns auc1, auc2, z, two-sided p."""
    y = np.asarray(y).astype(int); pos, neg = y == 1, y == 0; m, n = int(pos.sum()), int(neg.sum())
    aucs, V10, V01 = [], [], []
    for s in (np.asarray(s1, float), np.asarray(s2, float)):
        X, Y = s[pos], s[neg]
        tx, ty, tz = _midrank(X), _midrank(Y), _midrank(np.r_[X, Y])
        aucs.append((tz[:m].sum() - m * (m + 1) / 2) / (m * n))
        V10.append((tz[:m] - tx) / n); V01.append(1.0 - (tz[m:] - ty) / m)
    S10, S01 = np.cov(np.vstack(V10)), np.cov(np.vstack(V01))
    var = (S10[0, 0] + S10[1, 1] - 2 * S10[0, 1]) / m + (S01[0, 0] + S01[1, 1] - 2 * S01[0, 1]) / n
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return aucs[0], aucs[1], float(z), float(2 * sst.norm.sf(abs(z)))

def mcnemar_exact(y, pred1, pred2):
    """Exact McNemar test on the discordant pairs. Returns (only 1 right, only 2 right, two-sided p)."""
    y, pred1, pred2 = map(np.asarray, (y, pred1, pred2))
    n12 = int(((pred1 == y) & (pred2 != y)).sum()); n21 = int(((pred1 != y) & (pred2 == y)).sum())
    p = sst.binomtest(min(n12, n21), n12 + n21, 0.5).pvalue if n12 + n21 else 1.0
    return n12, n21, float(p)

def f1_pos_fast(y, p):
    tp = np.sum((p == 1) & (y == 1)); fp = np.sum((p == 1) & (y == 0)); fn = np.sum((p == 0) & (y == 1))
    return 2 * tp / max(2 * tp + fp + fn, 1)

def acc_fast(y, p):
    return float(np.mean(y == p))

def macro_f1_fast(y, p, k=None):
    k = int(max(y.max(), p.max()) + 1) if k is None else k
    cm = np.zeros((k, k)); np.add.at(cm, (y, p), 1)
    tp = np.diag(cm); prec = tp / np.maximum(cm.sum(0), 1); rec = tp / np.maximum(cm.sum(1), 1)
    return float(np.mean(2 * prec * rec / np.maximum(prec + rec, 1e-12)))

def _strat_resample(y, rng):
    return np.concatenate([rng.choice(np.where(y == c)[0], np.sum(y == c)) for c in np.unique(y)])

def bootstrap_ci(y, pred, metric, B=5000, seed=0):
    """Stratified bootstrap of the test set: point value, 95% percentile interval, standard error."""
    y, pred = np.asarray(y), np.asarray(pred); rng = np.random.default_rng(seed)
    v = np.array([metric(y[i], pred[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, pred), np.percentile(v, [2.5, 97.5]), float(v.std(ddof=1)), v

def paired_bootstrap(y, pred_a, pred_b, metric, B=5000, seed=0):
    """Difference metric(a) - metric(b) on the same test profiles, with a paired stratified bootstrap."""
    y, a, b = map(np.asarray, (y, pred_a, pred_b)); rng = np.random.default_rng(seed)
    d = np.array([metric(y[i], a[i]) - metric(y[i], b[i]) for i in (_strat_resample(y, rng) for _ in range(B))])
    return metric(y, a) - metric(y, b), np.percentile(d, [2.5, 97.5]), float(d.std(ddof=1))

def corrected_resampled_t(diffs, n_train, n_test, alternative="two-sided"):
    """Nadeau and Bengio (2003) corrected resampled t-test for J repeated random splits:
    the variance of the mean difference is inflated by (1/J + n_test/n_train) because training
    sets overlap between splits. Returns mean difference, t, p."""
    d = np.asarray(diffs, float); J = len(d)
    if J < 2 or np.allclose(d, d[0]):
        return float(d.mean()) if J else np.nan, np.nan, np.nan
    t = d.mean() / np.sqrt((1.0 / J + n_test / n_train) * d.var(ddof=1))
    p = sst.t.sf(t, J - 1) if alternative == "greater" else 2 * sst.t.sf(abs(t), J - 1)
    return float(d.mean()), float(t), float(p)

def wilcoxon_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float)
    if np.all(d == 0) or len(d) < 2:
        return np.nan
    try:
        return float(sst.wilcoxon(d, alternative=alternative, zero_method="wilcox").pvalue)
    except ValueError:
        return np.nan

def sign_test_p(diffs, alternative="two-sided"):
    d = np.asarray(diffs, float); k, n = int((d > 0).sum()), int((d != 0).sum())
    return float(sst.binomtest(k, n, 0.5, alternative=alternative).pvalue) if n else np.nan

def seed_mean_ci(values, B=20000, seed=0):
    """Percentile interval for a mean over seeds, by resampling seeds (wide when there are few seeds)."""
    v = np.asarray(values, float); rng = np.random.default_rng(seed)
    return np.percentile(rng.choice(v, (B, len(v))).mean(1), [2.5, 97.5])

def min_detectable_difference(se_diff, alpha=0.05, power=0.80):
    """Smallest true difference a two-sided test at level alpha detects with the given power."""
    return (sst.norm.ppf(1 - alpha / 2) + sst.norm.ppf(power)) * se_diff

def fmt_p(p):
    return "n/a" if p is None or (isinstance(p, float) and np.isnan(p)) else ("<0.001" if p < 0.001 else "%.3f" % p)

## Features: embedding variants, composition and metadata

In [ ]:
def _unit(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)
def _r2(E, direction):
    d = _unit(direction); return _unit(E - (E * d).sum(-1, keepdims=True) * d)

def text_variant(v, E, P):
    """Averages over the sections a posting actually contains (Gulati et al.'s STE)."""
    W = P[..., None].astype(np.float32); k = np.maximum(P.sum(1, keepdims=True), 1)
    if v in ("V1", "V1c"): return (E * W).sum(1) / k                          # plain mean
    if v in ("V0", "V0c"): return ((E - TAG_E[None]) * W).sum(1) / k           # STE
    if v == "V5": return (_r2(E, TAG_E[None]) * W).sum(1) / k                  # tag-direction projection
    raise ValueError(v)

WORDS = np.array([[len(t.split()) for t in row] for row in TEXT], float)
COMP = np.c_[P_ALL.astype(float), WORDS / np.maximum(WORDS.sum(1, keepdims=True), 1), np.log1p(WORDS.sum(1))]  # composition

def meta_features(rows_fit, rows):
    """Posting metadata; category levels are learned on the training split only."""
    parts = [df[c].fillna(0).astype(float).values[rows, None] for c in ["telecommuting", "has_company_logo", "has_questions"]]
    parts += [df[c].fillna("").astype(str).str.strip().ne("").values[rows, None].astype(float)
              for c in ["salary_range", "department", "location"]]
    for c, top in [("employment_type", None), ("required_experience", None), ("required_education", None),
                   ("function", None), ("industry", 30)]:
        col = df[c].fillna("missing").astype(str)
        levels = col.iloc[rows_fit].value_counts().index.tolist(); levels = levels[:top] if top else levels
        v = col.values[rows]
        parts.append(np.stack([v == l for l in levels], 1).astype(float))
        parts.append((~np.isin(v, levels)).astype(float)[:, None])
    return np.hstack(parts)

def build_X(v, rows_tr, rows_te):
    Xtr = np.hstack([text_variant(v, E_ALL[rows_tr], P_ALL[rows_tr]), meta_features(rows_tr, rows_tr)])
    Xte = np.hstack([text_variant(v, E_ALL[rows_te], P_ALL[rows_te]), meta_features(rows_tr, rows_te)])
    if v.endswith("c"):
        Xtr, Xte = np.hstack([Xtr, COMP[rows_tr]]), np.hstack([Xte, COMP[rows_te]])
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr).astype(np.float32), sc.transform(Xte).astype(np.float32)

# Proposition 1 does not apply here: the subtracted term is the mean tag of the sections present,
# so it is constant within a presence pattern and differs between patterns
D = text_variant("V0", E_ALL, P_ALL) - text_variant("V1", E_ALL, P_ALL)
within = max(float(np.abs(D[PATTERN == p] - D[PATTERN == p][0]).max()) for p in set(PATTERN))
between = len({tuple(np.round(D[PATTERN == p][0], 3)) for p in set(PATTERN)})
print("tag term: largest spread within a presence pattern %.1e (float rounding only); distinct values across the %d patterns: %d"
      % (within, len(set(PATTERN)), between))
for pat, g in pd.Series(Y_ALL).groupby(PATTERN):
    print("   pattern %s  postings %5d  fraudulent %5.1f%%" % (pat, len(g), 100 * g.mean()))

## Search and evaluation

In [ ]:
def _fold_score(tmpl, X, y, a, b, orness):
    import warnings; warnings.filterwarnings("ignore")        # worker processes do not inherit the filter
    try:
        est = clone(tmpl).fit(X[a], y[a])
        return fuzzy_multi_metric(y[b], est.predict(X[b]), est.predict_proba(X[b])[:, 1], orness=orness)
    except Exception:
        return None

def make_cv_fitness_par(X, y, seed, n_splits=5, orness=0.30, gap_penalty=0.50):
    """Same computation as make_cv_fitness() of the paper; the five folds and the training refit run in parallel."""
    jobs = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X, y))
    jobs.append((np.arange(len(y)), np.arange(len(y))))
    def fitness(params):
        try:
            tmpl = build_estimator(params)
        except Exception:
            return 0.0
        out = Parallel(n_jobs=N_JOBS)(delayed(_fold_score)(tmpl, X, y, a, b, orness) for a, b in jobs)
        folds = [0.0 if s is None else s for s in out[:-1]]; mean_cv = float(np.mean(folds))
        train_full = mean_cv if out[-1] is None else out[-1]
        return fuzzy_owa(folds, orness=orness) - gap_penalty * max(0.0, train_full - mean_cv)
    return fitness

def save_pred(name, **arrays): np.savez_compressed(os.path.join(PRED_DIR, name + ".npz"), **arrays)
def load_pred(name):
    p = os.path.join(PRED_DIR, name + ".npz"); return dict(np.load(p)) if os.path.exists(p) else None
def done_row(path, **key):
    if not os.path.exists(path): return None
    d = pd.read_csv(path)
    for k, v in key.items():
        if k not in d.columns: return None
        d = d[d[k] == v]
    return d.iloc[0].to_dict() if len(d) else None
def append_row(path, row):
    d = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    _atomic_csv(pd.concat([d, pd.DataFrame([row])], ignore_index=True), path)

EMS_OUT = os.path.join(OUT_DIR, "results_emscad.csv")
HGS_KW = dict(pop_size=12, max_iter=18, early_stop_patience=4)

def run_emscad(protocol, variant, seed, strategy="HGS", budget=None):
    hit = done_row(EMS_OUT, protocol=protocol, variant=variant, seed=seed, strategy=strategy)
    if hit:
        print("%-5s %-3s seed %-4d %-6s already done - skipped" % (protocol, variant, seed, strategy)); return hit
    rows = ROWSETS[protocol]; y = Y_ALL[rows]
    tr, te = train_test_split(np.arange(len(rows)), test_size=0.30, random_state=seed, stratify=y)
    Xtr, Xte = build_X(variant, rows[tr], rows[te])
    cache = os.path.join(CACHE_DIR, "emscad__%s_%s_s%d.pkl" % (protocol, variant, seed))   # shared by both searches
    fit = PersistentCounted(make_cv_fitness_par(Xtr, y[tr], seed), cache)
    t0 = time.time()
    if strategy == "HGS":
        bp, bs, _ = hgs_search(LR_SPACE, fit, seed=seed, verbose=False, **HGS_KW)
    else:
        bp, bs = random_search(fit, int(budget), seed)
    m = build_estimator(bp).fit(Xtr, y[tr]); proba = m.predict_proba(Xte)[:, 1]
    r = evaluate(y[te], proba)
    in_train = set(KEY.values[rows[tr]]); dup = np.array([k in in_train for k in KEY.values[rows[te]]])
    r.update(protocol=protocol, variant=variant, seed=seed, strategy=strategy, cv_fitness=bs, n_models=fit.n,
             n_train=len(tr), n_features=Xtr.shape[1], test_with_copy_in_train=float(dup.mean()),
             fraud_test_with_copy_in_train=float(dup[y[te] == 1].mean()),
             seconds=round(time.time() - t0, 1), params_fitted=json.dumps(dict(canonical(bp))))
    save_pred("emscad__%s_%s_s%d_%s" % (protocol, variant, seed, strategy),
              y=y[te], proba=proba, pred=(proba >= 0.5).astype(int), test_rows=rows[te])
    append_row(EMS_OUT, r)
    print("%-5s %-3s seed %-4d %-6s models=%-3d F1+=%.4f P=%.4f R=%.4f AUC=%.4f PR-AUC=%.4f (%.0f min)" % (
        protocol, variant, seed, strategy, r["n_models"], r["f1_pos"], r["precision_pos"], r["recall_pos"],
        r["roc_auc"], r["pr_auc"], r["seconds"] / 60))
    return r

## Plan and timing estimate

In [ ]:
SEEDS_14 = [42, 7, 123]
PLAN = [("dedup", "V0", "HGS"), ("dedup", "V1", "HGS"), ("dedup", "V5", "HGS"),   # composition test and projection
        ("dedup", "V1c", "HGS"),                                                   # plain mean + explicit composition
        ("full", "V0", "HGS"), ("full", "V5", "HGS"),                              # duplicate leakage
        ("dedup", "V5", "Random")]                                                 # search strategy
# ---- quick timing estimate on this machine, before committing to the plan ----
rows = ROWSETS["dedup"]; y = Y_ALL[rows]
tr, _ = train_test_split(np.arange(len(rows)), test_size=0.30, random_state=42, stratify=y)
Xtr, _ = build_X("V0", rows[tr], rows[tr[:10]]); f = make_cv_fitness_par(Xtr, y[tr], 42)
t0 = time.time()
for p in [dict(C=0.5, penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=1000, l1_ratio=0.5),
          dict(C=0.5, penalty="l1", solver="liblinear", class_weight="balanced", max_iter=1000, l1_ratio=0.5),
          dict(C=0.5, penalty="l1", solver="saga", class_weight=None, max_iter=500, l1_ratio=0.5)]:
    f(p)
per = (time.time() - t0) / 3
print("about %.0f s per configuration here -> roughly %.0f-%.0f min per HGS run (15-30 configurations), %.1f-%.1f h for the plan" % (
    per, per * 15 / 60, per * 30 / 60, per * 15 * len(PLAN) * len(SEEDS_14) / 3600, per * 30 * len(PLAN) * len(SEEDS_14) / 3600))

## Run (resumable)

In [ ]:
for seed in SEEDS_14:
    for protocol, v, strategy in PLAN:
        if strategy == "HGS":
            run_emscad(protocol, v, seed)
        else:
            ref = done_row(EMS_OUT, protocol=protocol, variant=v, seed=seed, strategy="HGS")
            if ref: run_emscad(protocol, v, seed, "Random", budget=ref["n_models"])

## Analysis

In [ ]:
M = pd.read_csv(EMS_OUT); L = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); L.append(s)
say("=== Results, mean +- SD over splits (positive class = fraudulent) ===")
cols = ["f1_pos", "precision_pos", "recall_pos", "roc_auc", "pr_auc", "mcc", "f1_macro", "accuracy"]
say(M.groupby(["protocol", "strategy", "variant"])[cols].agg(lambda v: "%.4f +- %.4f" % (v.mean(), v.std(ddof=1) if len(v) > 1 else 0)).to_string())

def paired(protocol, a, b, strat_a="HGS", strat_b="HGS", title=""):
    diffs = []; say("--- %s ---" % title)
    for s in SEEDS_14:
        A, Bp = load_pred("emscad__%s_%s_s%d_%s" % (protocol, a, s, strat_a)), load_pred("emscad__%s_%s_s%d_%s" % (protocol, b, s, strat_b))
        if A is None or Bp is None: continue
        assert np.array_equal(A["test_rows"], Bp["test_rows"])
        d, ci, se = paired_bootstrap(A["y"], A["pred"], Bp["pred"], f1_pos_fast, B=3000, seed=s)
        aa, ab, _, p_auc = delong_test(A["y"], A["proba"], Bp["proba"]); n12, n21, p_mc = mcnemar_exact(A["y"], A["pred"], Bp["pred"])
        diffs.append(d)
        say("seed %-4d dF1 %+.4f [%+.4f, %+.4f] | AUC %.4f vs %.4f, DeLong p %s | McNemar %d/%d p %s" % (
            s, d, ci[0], ci[1], aa, ab, fmt_p(p_auc), n12, n21, fmt_p(p_mc)))
    if len(diffs) >= 2:
        n_te = len(A["y"]); m, t, p = corrected_resampled_t(diffs, int(round(n_te * 0.7 / 0.3)), n_te)
        say("mean dF1 %+.4f | corrected resampled t two-sided p %s" % (m, fmt_p(p)))
        return m, p
    return None, None

say("\n=== Composition hypothesis (paper, Discussion): the tag term helps only through section composition ===")
m1, p1 = paired("dedup", "V0", "V1", title="STE (V0) vs plain mean (V1): does subtracting the tag help when sections vary?")
m2, p2 = paired("dedup", "V1c", "V0", title="plain mean + explicit composition (V1c) vs STE (V0): does composition explain it?")
if m1 is not None and m2 is not None:
    step1 = m1 > 0 and p1 == p1 and p1 < 0.05
    step2 = m2 >= 0 or not (p2 == p2 and p2 < 0.05)
    say("Prediction: STE beats the plain mean (significantly) AND explicit composition features do at least as well as STE.")
    say("  STE > plain mean: %s | composition features >= STE: %s -> %s" % (
        "yes" if step1 else "no", "yes" if step2 else "no",
        "SUPPORTED" if (step1 and step2) else ("NOT SUPPORTED: the tag term adds nothing here" if not step1
                                               else "PARTLY: STE helps, but not only through composition")))
say("\n=== Tag-direction projection ===")
paired("dedup", "V5", "V0", title="projection (V5) vs STE (V0)")
say("\n=== Search strategy ===")
paired("dedup", "V5", "V5", "HGS", "Random", title="HGS vs matched-budget random search (V5)")
say("\n=== Duplicate leakage: full data vs one copy per duplicate group (different test sets, not paired) ===")
for v in ["V0", "V5"]:
    f, d = M[(M.protocol == "full") & (M.variant == v) & (M.strategy == "HGS")], M[(M.protocol == "dedup") & (M.variant == v) & (M.strategy == "HGS")]
    if len(f) and len(d):
        say("%s: F1 %.4f (full) vs %.4f (dedup), difference %+.4f | ROC area %.4f vs %.4f | test postings with a copy in training: %.1f%% (fraudulent: %.1f%%)" % (
            v, f.f1_pos.mean(), d.f1_pos.mean(), f.f1_pos.mean() - d.f1_pos.mean(), f.roc_auc.mean(), d.roc_auc.mean(),
            100 * f.test_with_copy_in_train.mean(), 100 * f.fraud_test_with_copy_in_train.mean()))
say("\n=== Published results on EMSCAD (verify each against the paper before citing) ===")
say("Vidros et al. 2017, Future Internet 9(1):6 - full imbalanced data, random forest: accuracy 0.895, "
    "fraudulent precision 0.282, recall 0.751 (verified from the paper, Sec. 6.4)")
say("Amaar et al. 2022, Neural Processing Letters - extra trees, TF-IDF, ADASYN: accuracy 0.999 (abstract; "
    "the order of oversampling and splitting could not be verified)")
say("Our comparable figures: the 'full' protocol (same data as the literature) and the 'dedup' protocol (leakage-free).")
open(os.path.join(OUT_DIR, "emscad_report.txt"), "w").write("\n".join(L))